# Armored Core 6 — Relation Verifier (with special tokens) — Colab / GPU

Trains the **relation verifier** (BERT + `[REL] [/REL] [E1] [/E1] [E2] [/E2]` special tokens,
binary `VALID` / `INVALID`). Mirrors the professor's
`logistics_verifier_pipeline_with_special_tokens_colab.ipynb`.

**How to use on Google Colab:** Runtime → Change runtime type → **GPU**, then run all.
Upload `ac6_verifier_train.jsonl`, `ac6_verifier_val.jsonl`, `ac6_verifier_test.jsonl`
(produced by `01_build_corpus.py`) when prompted, or mount Drive.

You can also run this **locally** with the script `03_train_verifier.py` — same recipe.

In [ ]:
!pip -q install transformers torch tqdm scikit-learn seaborn matplotlib

In [ ]:
# change only this to try another encoder
MODEL_NAME = "bert-base-uncased"
EPOCHS = 4
BATCH_SIZE = 16
LR = 2e-5
MAX_LEN = 160
SPECIAL = ["[REL]", "[/REL]", "[E1]", "[/E1]", "[E2]", "[/E2]"]

In [ ]:
# Colab: upload the three verifier JSONL files (skip if already present)
import os
paths = {n: f"ac6_verifier_{n}.jsonl" for n in ("train", "val", "test")}
if not all(os.path.exists(p) for p in paths.values()):
    try:
        from google.colab import files
        print("Upload ac6_verifier_train.jsonl, ac6_verifier_val.jsonl, ac6_verifier_test.jsonl")
        up = files.upload()
    except Exception as e:
        print("Not on Colab. Put the JSONL files next to this notebook. (", e, ")")
print({n: os.path.exists(p) for n, p in paths.items()})

In [ ]:
import json
def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
train = load_jsonl(paths["train"]); val = load_jsonl(paths["val"]); test = load_jsonl(paths["test"])
labels = sorted({r["label"] for r in train + val + test})
label2id = {l: i for i, l in enumerate(labels)}; id2label = {i: l for l, i in label2id.items()}
print("sizes:", len(train), len(val), len(test), "| labels:", labels)

In [ ]:
import torch, numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL})

def pretokenize_rows(rows, tokenizer, label2id, max_len):
    enc = tokenizer([r["text"] for r in rows], truncation=True, padding="max_length",
                    max_length=max_len, return_tensors="pt")
    y = torch.tensor([label2id[r["label"]] for r in rows])
    return enc, y

class DS(Dataset):
    def __init__(self, enc, y): self.enc, self.y = enc, y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        d = {k: v[i] for k, v in self.enc.items()}; d["labels"] = self.y[i]; return d

tr = DataLoader(DS(*pretokenize_rows(train, tokenizer, label2id, MAX_LEN)), batch_size=BATCH_SIZE, shuffle=True)
va = DataLoader(DS(*pretokenize_rows(val,   tokenizer, label2id, MAX_LEN)), batch_size=64)
te = DataLoader(DS(*pretokenize_rows(test,  tokenizer, label2id, MAX_LEN)), batch_size=64)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(labels))
model.config.id2label = id2label; model.config.label2id = label2id
model.resize_token_embeddings(len(tokenizer)); model.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR)
total = len(tr) * EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)

def evaluate(loader):
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for b in loader:
            b = {k: v.to(device) for k, v in b.items()}
            logits = model(**{k: v for k, v in b.items() if k != "labels"}).logits
            ps += logits.argmax(-1).cpu().tolist(); ys += b["labels"].cpu().tolist()
    return ys, ps

BEST_OUT = "ac6_verifier_best"; best = -1; history = []
for ep in range(1, EPOCHS + 1):
    model.train(); tot = 0.0
    for b in tr:
        b = {k: v.to(device) for k, v in b.items()}
        opt.zero_grad(); loss = model(**b).loss; loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); tot += loss.item()
    ys, ps = evaluate(va); acc = float(np.mean([y == p for y, p in zip(ys, ps)]))
    history.append((ep, tot / len(tr), acc)); print(f"epoch {ep}: loss={tot/len(tr):.4f} val_acc={acc:.4f}")
    if acc >= best:
        best = acc; model.save_pretrained(BEST_OUT); tokenizer.save_pretrained(BEST_OUT)
print("best val acc:", best)

## Test metrics, accuracy curve and confusion matrix (the precision graphs)

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

best_model = AutoModelForSequenceClassification.from_pretrained(BEST_OUT).to(device); best_model.eval()
def eval_best(loader):
    ys, ps = [], []
    with torch.no_grad():
        for b in loader:
            b = {k: v.to(device) for k, v in b.items()}
            logits = best_model(**{k: v for k, v in b.items() if k != "labels"}).logits
            ps += logits.argmax(-1).cpu().tolist(); ys += b["labels"].cpu().tolist()
    return ys, ps
ys, ps = eval_best(te)
print(classification_report(ys, ps, target_names=labels, digits=4))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([h[0] for h in history], [h[2] for h in history], marker="o")
ax[0].set_title("Validation accuracy"); ax[0].set_xlabel("epoch"); ax[0].set_ylim(0, 1.02); ax[0].grid(alpha=.3)
cm = confusion_matrix(ys, ps)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax[1])
ax[1].set_title("Test confusion matrix"); ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("Actual")
plt.tight_layout(); plt.show()

In [ ]:
# query the freshly trained verifier on a manual claim
def ask(rel, subj, obj):
    phrase = {"pilots":"pilots","kills":"kills","worksFor":"works for"}.get(rel, rel)
    text = f"[REL] {rel} [/REL] A [E1]{subj}[/E1] {phrase} a [E2]{obj}[/E2]."
    with torch.no_grad():
        logits = best_model(**{k: v.to(device) for k, v in tokenizer(text, return_tensors="pt").items()}).logits
    return id2label[int(logits.argmax(-1))]
print("character pilots armored core ->", ask("pilots", "character", "armored core"))
print("place pilots armored core     ->", ask("pilots", "place", "armored core"))

## Why this JSONL input is valid for the verifier

Each example is a single string `"[REL] <relation> [/REL] A [E1]<subject>[/E1] <phrase> a [E2]<object>[/E2]."`
with a `VALID` / `INVALID` label. The special tokens isolate the **candidate relation** and the
**two argument slots**, so the encoder learns the ontology's domain/range compatibility instead of
surface wording. `INVALID` rows reuse the same sentence but swap in a wrong candidate relation.